<a href="https://colab.research.google.com/github/AHMADMALIK1376/Artificial-Intelligence/blob/main/LAB_NO_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

tree = {
 'A': ['B', 'C'],
 'B': ['D', 'E'],
 'C': ['F', 'G'],
 'D': ['H', 'I'],
 'E': ['J'],
 'F': ['K'],
 'G': [],
 'H': [], 'I': [], 'J': [], 'K': []
}

def dls(node, goal, limit, visited_order):
 """Recursive Depth-Limited Search.
 Returns 'found', 'cutoff' or 'failure'."""
 visited_order.append(node)
 if node == goal:
  return 'found'
 if limit == 0:
  return 'cutoff'
 cutoff_occurred = False
 for child in tree[node]:
  res = dls(child, goal, limit - 1, visited_order)
  if res == 'found':
   return 'found'
  if res == 'cutoff':
   cutoff_occurred = True
 return 'cutoff' if cutoff_occurred else 'failure'

def iddfs(start, goal, max_depth=10):
 """Iterative Deepening: call DLS with limit 0, 1, 2, ..."""
 for depth in range(max_depth + 1):
  order = []
  res = dls(start, goal, depth, order)
  print(f"Depth limit {depth}: visited {' '.join(order):<24} -> {res}")
  if res == 'found':
   return depth
  if res == 'failure':
   return None
 return None

print("DLS with limit 2, goal = 'J'")
order = []
print("Result:", dls('A', 'J', 2, order), "| visited:", ' '.join(order))

print("\nIDDFS, goal = 'J'")
d = iddfs('A', 'J')
print("Goal found at depth", d)

# ----- Visualise the tree, colouring nodes by depth -----
G = nx.DiGraph([(p, c) for p, cs in tree.items() for c in cs])

# simple layered layout: each leaf gets its own x-slot, parents sit above children
pos, next_x = {}, [0]
def layout(n, d=0):
 kids = tree[n]
 if not kids:
  pos[n] = (next_x[0], -d); next_x[0] += 1
 else:
  for k in kids: layout(k, d + 1)
  pos[n] = (sum(pos[k][0] for k in kids) / len(kids), -d)
layout('A')

# Define colors
NAVY = '#0E367B'
GOLD = '#FFD700'  # Highlight color for the path

# Find path to 'J'
path = ['A', 'B', 'E', 'J']
path_edges = list(zip(path[:-1], path[1:]))

# Set node colors (nodes on path get highlighted)
node_colors = [GOLD if n in path else 'white' for n in G.nodes]
node_edge_colors = [NAVY if n in path else '#CCCCCC' for n in G.nodes]

# Set edge colors (edges on path get highlighted)
edge_colors = [GOLD if e in path_edges else '#CCCCCC' for e in G.edges]
edge_widths = [3.5 if e in path_edges else 1.5 for e in G.edges]

plt.figure(figsize=(6, 3.6))

# Draw edges
nx.draw_networkx_edges(G, pos, edgelist=G.edges, edge_color=edge_colors, width=edge_widths, arrows=False)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=700, edgecolors=node_edge_colors, linewidths=2)

# Draw labels
labels_font_color = {n: NAVY for n in G.nodes}
nx.draw_networkx_labels(G, pos, font_weight='bold', font_color=labels_font_color)

plt.title("Search Tree (Path to goal J highlighted in gold)")
plt.savefig('lab04_tree.png', dpi=130, bbox_inches='tight')
print("Tree with highlighted path saved to lab04_tree.png")

DLS with limit 2, goal = 'J'
Result: cutoff | visited: A B D E C F G

IDDFS, goal = 'J'
Depth limit 0: visited A                        -> cutoff
Depth limit 1: visited A B C                    -> cutoff
Depth limit 2: visited A B D E C F G            -> cutoff
Depth limit 3: visited A B D H I E J            -> found
Goal found at depth 3
Tree with highlighted path saved to lab04_tree.png


In [10]:
from collections import deque
import heapq

# ---------- Grid maze: S = start, G = goal, # = wall ----------
MAZE = ["S . . . . # #",
        "# # . # # . .",
        ". . . # . . #",
        ". # . . . . #",
        ". . . . # . G"]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell
    for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1)]:  # Up, Right, Down, Left
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def bfs(start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        node = frontier.popleft()  # FIFO queue
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded
        for nb in neighbours(node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
    return None, expanded

def dfs(start, goal):
    frontier = [start]
    parent = {start: None}
    expanded = 0
    visited = set()
    while frontier:
        node = frontier.pop()  # LIFO stack
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded
        for nb in neighbours(node):
            if nb not in visited:
                parent[nb] = node
                frontier.append(nb)
    return None, expanded

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

S, G = find('S'), find('G')
print(f"{'Algorithm':<6}{'Path length':>13}{'Nodes expanded':>17}")
for name, algo in [('BFS', bfs), ('DFS', dfs)]:
    path, exp = algo(S, G)
    print(f"{name:<6}{len(path)-1:>13}{exp:>17}")

# ---------- Uniform Cost Search on a weighted graph ----------
graph = {
    'Islamabad': {'Rawalpindi': 15, 'Murree': 60, 'Attock': 80},
    'Rawalpindi': {'Islamabad': 15, 'Chakwal': 90, 'Jhelum': 110},
    'Murree': {'Islamabad': 60, 'Abbottabad': 65},
    'Attock': {'Islamabad': 80, 'Peshawar': 100},
    'Chakwal': {'Rawalpindi': 90, 'Lahore': 230},
    'Jhelum': {'Rawalpindi': 110, 'Gujranwala': 110},
    'Gujranwala': {'Jhelum': 110, 'Lahore': 70},
    'Abbottabad': {'Murree': 65},
    'Peshawar': {'Attock': 100},
    'Lahore': {'Chakwal': 230, 'Gujranwala': 70},
}

def ucs(graph, start, goal):
    frontier = [(0, start, [start])]  # (path cost, node, path)
    explored = set()
    expanded = 0
    while frontier:
        cost, node, path = heapq.heappop(frontier)  # lowest g(n) first
        if node in explored:
            continue
        explored.add(node)
        expanded += 1
        if node == goal:
            return path, cost, expanded
        for nb, w in graph[node].items():
            if nb not in explored:
                heapq.heappush(frontier, (cost + w, nb, path + [nb]))
    return None, float('inf'), expanded

path, cost, exp = ucs(graph, 'Islamabad', 'Lahore')
print("\nUCS path :", ' -> '.join(path))
print("UCS cost :", cost, "km | nodes expanded:", exp)

Algorithm  Path length   Nodes expanded
BFS              10               23
DFS              16               18

UCS path : Islamabad -> Rawalpindi -> Jhelum -> Gujranwala -> Lahore
UCS cost : 305 km | nodes expanded: 10


In [11]:
from collections import deque

MAZE = ["S . . . . # #",
        "# # . # # . .",
        ". . . # . . #",
        ". # . . . . #",
        ". . . . # . G"]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell
    for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1)]:  # Up, Right, Down, Left
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    max_frontier = 1
    while frontier:
        node = frontier.popleft()  # FIFO queue
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def dfs(start, goal):
    frontier = [start]
    parent = {start: None}
    expanded = 0
    visited = set()
    max_frontier = 1
    while frontier:
        node = frontier.pop()  # LIFO stack
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in visited:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

S, G = find('S'), find('G')
print(f"{'Algorithm':<10}{'Path length':>13}{'Nodes expanded':>17}{'Max frontier':>15}")
for name, algo in [('BFS', bfs), ('DFS', dfs)]:
    path, exp, mf = algo(S, G)
    print(f"{name:<10}{len(path)-1:>13}{exp:>17}{mf:>15}")

Algorithm   Path length   Nodes expanded   Max frontier
BFS                  10               23              5
DFS                  16               18              6


In [ ]:
from collections import deque

MAZE = ["S . . . . # #",
        "# # . # # . .",
        ". . . # . . #",
        ". # . . . . #",
        ". . . . # . G"]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

def find(ch):     for r in range(ROWS):         for c in range(COLS):             if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell     for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1)]:  # Up, Right, Down, Left         nr, nc = r + dr, c + dc         if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def build_path(parent, node):     path = []     while node is not None:         path.append(node)         node = parent[node]     return path[::-1]

def bfs(start, goal):     frontier = deque([start])     parent = {start: None}     while frontier:
        node = frontier.popleft()         if node == goal:
            return build_path(parent, goal)         for nb in neighbours(node):             if nb not in parent:                 parent[nb] = node                 frontier.append(nb)     return None

def dfs(start, goal):     frontier = [start]     parent = {start: None}     visited = set()     while frontier:
        node = frontier.pop()         if node in visited:             continue         visited.add(node)         if node == goal:
            return build_path(parent, goal)         for nb in neighbours(node):             if nb not in visited:                 parent[nb] = node                 frontier.append(nb)     return None

def print_maze(path, label):
    on_path = set(path)     print(label)     for r in range(ROWS):
        row = []         for c in range(COLS):
            ch = grid[r][c]             if (r, c) in on_path and ch not in ('S', 'G'):
                ch = '*'             row.append(ch)         print(' '.join(row))     print()

S, G = find('S'), find('G') print_maze(bfs(S, G), "BFS path:") print_maze(dfs(S, G), "DFS path:")



In [9]:
from collections import deque

MAZE = ["S . . . . # #",
        "# # . # # . .",
        ". . . # . . #",
        ". # . . . . #",
        ". . . . # . G"]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

ORDER_OLD = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # Up, Right, Down, Left
ORDER_NEW = [(0, -1), (1, 0), (0, 1), (-1, 0)]   # Left, Down, Right, Up
ORDER = ORDER_OLD

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell
    for dr, dc in ORDER:
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    max_frontier = 1
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def dfs(start, goal):
    frontier = [start]
    parent = {start: None}
    expanded = 0
    visited = set()
    max_frontier = 1
    while frontier:
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in visited:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def print_maze(path, label):
    on_path = set(path) if path else set()
    print(label)
    for r in range(ROWS):
        print(' '.join('*' if (r, c) in on_path and grid[r][c] not in ('S', 'G')
                       else grid[r][c] for c in range(COLS)))
    print()

S, G = find('S'), find('G')

for label, order in [("Order: Up, Right, Down, Left", ORDER_OLD),
                     ("Order: Left, Down, Right, Up", ORDER_NEW)]:
    ORDER = order
    print(label)
    print(f"{'Algorithm':<10}{'Path length':>13}{'Nodes expanded':>17}{'Max frontier':>15}")
    paths = {}
    for name, algo in [('BFS', bfs), ('DFS', dfs)]:
        path, exp, mf = algo(S, G)
        paths[name] = path
        print(f"{name:<10}{len(path)-1:>13}{exp:>17}{mf:>15}")
    print()
    print_maze(paths['BFS'], "BFS path:")
    print_maze(paths['DFS'], "DFS path:")

Order: Up, Right, Down, Left
Algorithm   Path length   Nodes expanded   Max frontier
BFS                  10               23              5
DFS                  16               18              6

BFS path:
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G

DFS path:
S * * . . # #
# # * # # . .
* * * # . . #
* # . * * * #
* * * * # * G

Order: Left, Down, Right, Up
Algorithm   Path length   Nodes expanded   Max frontier
BFS                  10               22              4
DFS                  12               17              6

BFS path:
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G

DFS path:
S * * . . # #
# # * # # . .
. . * # * * #
. # * * * * #
. . . . # * G



In [4]:
from collections import deque

MAZE = ["S . . . . # #",
        "# # . # # . .",
        ". . . # . . #",
        ". # . . . . #",
        ". . . . # . G"]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

ORDER_OLD = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # Up, Right, Down, Left
ORDER_NEW = [(0, -1), (1, 0), (0, 1), (-1, 0)]   # Left, Down, Right, Up
ORDER = ORDER_OLD

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell
    for dr, dc in ORDER:
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    max_frontier = 1
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def dfs(start, goal):
    frontier = [start]
    parent = {start: None}
    expanded = 0
    visited = set()
    max_frontier = 1
    while frontier:
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node):
            if nb not in visited:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def print_maze(path, label):
    on_path = set(path) if path else set()
    print(label)
    for r in range(ROWS):
        print(' '.join('*' if (r, c) in on_path and grid[r][c] not in ('S', 'G')
                       else grid[r][c] for c in range(COLS)))
    print()

S, G = find('S'), find('G')

for label, order in [("Order: Up, Right, Down, Left", ORDER_OLD),
                     ("Order: Left, Down, Right, Up", ORDER_NEW)]:
    ORDER = order
    print(label)
    print(f"{'Algorithm':<10}{'Path length':>13}{'Nodes expanded':>17}{'Max frontier':>15}")
    paths = {}
    for name, algo in [('BFS', bfs), ('DFS', dfs)]:
        path, exp, mf = algo(S, G)
        paths[name] = path
        print(f"{name:<10}{len(path)-1:>13}{exp:>17}{mf:>15}")
    print()
    print_maze(paths['BFS'], "BFS path:")
    print_maze(paths['DFS'], "DFS path:")

Order: Up, Right, Down, Left
Algorithm   Path length   Nodes expanded   Max frontier
BFS                  10               23              5
DFS                  16               18              6

BFS path:
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G

DFS path:
S * * . . # #
# # * # # . .
* * * # . . #
* # . * * * #
* * * * # * G

Order: Left, Down, Right, Up
Algorithm   Path length   Nodes expanded   Max frontier
BFS                  10               22              4
DFS                  12               17              6

BFS path:
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G

DFS path:
S * * . . # #
# # * # # . .
. . * # * * #
. # * * * * #
. . . . # * G



In [12]:
import heapq

graph = {
    'Islamabad': {'Rawalpindi': 15, 'Murree': 60, 'Attock': 80},
    'Rawalpindi': {'Islamabad': 15, 'Chakwal': 90, 'Jhelum': 110},
    'Murree': {'Islamabad': 60, 'Abbottabad': 65},
    'Attock': {'Islamabad': 80, 'Peshawar': 100},
    'Chakwal': {'Rawalpindi': 90, 'Lahore': 230},
    'Jhelum': {'Rawalpindi': 110, 'Gujranwala': 110},
    'Gujranwala': {'Jhelum': 110, 'Lahore': 70},
    'Abbottabad': {'Murree': 65},
    'Peshawar': {'Attock': 100},
    'Lahore': {'Chakwal': 230, 'Gujranwala': 70},
}

def ucs(graph, start, goal):
    frontier = [(0, start, [start])]  # (path cost, node, path)
    explored = set()
    expanded = 0
    while frontier:
        cost, node, path = heapq.heappop(frontier)  # lowest g(n) first
        if node in explored:
            continue
        explored.add(node)
        expanded += 1
        if node == goal:
            return path, cost, expanded
        for nb, w in graph[node].items():
            if nb not in explored:
                heapq.heappush(frontier, (cost + w, nb, path + [nb]))
    return None, float('inf'), expanded

def add_edge(g, a, b, w):
    # roads are two-way
    g[a][b] = w
    g[b][a] = w

path, cost, exp = ucs(graph, 'Islamabad', 'Lahore')
print("BEFORE  UCS path :", ' -> '.join(path))
print("        UCS cost :", cost, "km | nodes expanded:", exp)

add_edge(graph, 'Islamabad', 'Chakwal', 120)
add_edge(graph, 'Chakwal', 'Gujranwala', 150)

path, cost, exp = ucs(graph, 'Islamabad', 'Lahore')
print("\nAFTER   UCS path :", ' -> '.join(path))
print("        UCS cost :", cost, "km | nodes expanded:", exp)

BEFORE  UCS path : Islamabad -> Rawalpindi -> Jhelum -> Gujranwala -> Lahore
        UCS cost : 305 km | nodes expanded: 10

AFTER   UCS path : Islamabad -> Rawalpindi -> Jhelum -> Gujranwala -> Lahore
        UCS cost : 305 km | nodes expanded: 10


In [13]:
import os
from collections import deque
import heapq

# Default maze (written to maze.txt automatically if the file does not exist)
DEFAULT_MAZE = """S . . m m . . . . .
. # . # m # # # . # . # . . . . m . . .
. # # # # . # # # .
. . . m m . . . # .
# # . # # # # . . .
. . . . m m . . # .
. # # # # . # # # . . m m m . . . . . .
. . . . . # # # . G
"""

# Works in notebooks (Jupyter/Colab) AND in normal .py scripts
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))   # running as a .py script
except NameError:
    BASE_DIR = os.getcwd()                                  # running in a notebook

FILE = os.path.join(BASE_DIR, "maze.txt")

if not os.path.exists(FILE):
    with open(FILE, "w") as f:
        f.write(DEFAULT_MAZE)
    print(f"(maze.txt not found - created a default one at {FILE})\n")

def load_maze(filename):
    with open(filename) as f:
        return [line.split() for line in f if line.strip()]

grid = load_maze(FILE)
ROWS, COLS = len(grid), len(grid[0])

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)
    raise ValueError(f"'{ch}' not found in maze.txt")

def neighbours(cell):
    r, c = cell
    for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1)]:  # Up, Right, Down, Left
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def step_cost(cell):
    return 5 if grid[cell[0]][cell[1]] == 'm' else 1   # cost of ENTERING the cell

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def path_cost(path):
    return sum(step_cost(cell) for cell in path[1:])

def bfs(start, goal):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded
        for nb in neighbours(node):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
    return None, expanded

def ucs(start, goal):
    frontier = [(0, start, [start])]
    explored = set()
    expanded = 0
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored:
            continue
        explored.add(node)
        expanded += 1
        if node == goal:
            return path, expanded
        for nb in neighbours(node):
            if nb not in explored:
                heapq.heappush(frontier, (cost + step_cost(nb), nb, path + [nb]))
    return None, expanded

def print_maze(path, label):
    on_path = set(path) if path else set()
    print(label)
    for r in range(ROWS):
        print(' '.join('*' if (r, c) in on_path and grid[r][c] not in ('S', 'G') else grid[r][c] for c in range(COLS)))
    print()

S, G = find('S'), find('G')
bpath, bexp = bfs(S, G)
upath, uexp = ucs(S, G)

if bpath is None or upath is None:
    print("No path exists between S and G in this maze.")
else:
    print(f"{'Algorithm':<10}{'Moves':>8}{'Total cost':>12}{'Nodes expanded':>17}")
    print(f"{'BFS':<10}{len(bpath)-1:>8}{path_cost(bpath):>12}{bexp:>17}")
    print(f"{'UCS':<10}{len(upath)-1:>8}{path_cost(upath):>12}{uexp:>17}\n")
    print_maze(bpath, "BFS path:")
    print_maze(upath, "UCS path:")

(maze.txt not found - created a default one at /content/maze.txt)

Algorithm    Moves  Total cost   Nodes expanded
BFS             16          24               48
UCS             16          24               48

BFS path:
S . . m m . . . . .
* # . # m # # # . #
* # # # # . # # # .
* * * * * * * * # .
# # . # # # # * * *
. . . . m m . . # *
. # # # # . # # # *
. . . . . # # # . G

UCS path:
S . . m m . . . . .
* # . # m # # # . #
* # # # # . # # # .
* * * * * * * * # .
# # . # # # # * * *
. . . . m m . . # *
. # # # # . # # # *
. . . . . # # # . G

